# First-pass NER on the Italian corpus

Run the cells in order using a CPU runtime. The notebook downloads data from their original sources and runs rules plus spaCy. It does not run perplexity or an LLM. Save the output manually before disconnecting.


In [ ]:
import os
from pathlib import Path
import subprocess
import sys

repo = Path("/content/forensic-llm-graph")
if repo.exists():
    subprocess.run(["git", "pull", "--ff-only"], cwd=repo, check=True)
else:
    subprocess.run(["git", "clone", "https://github.com/AntoFratta/forensic-llm-graph.git", str(repo)], check=True)
os.chdir(repo)
subprocess.run(["git", "log", "-1", "--oneline"], check=True)
print("Python:", sys.version.split()[0])


In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)


In [ ]:
subprocess.run([sys.executable, "scripts/download_data.py"], check=True)


In [ ]:
subprocess.run([sys.executable, "scripts/prepare_data.py", "--overwrite"], check=True)

import json
from collections import Counter

documents = json.loads(Path("data/processed/documents.json").read_text(encoding="utf-8"))
counts = Counter(document["source"] for document in documents)
expected = {"redit": 126, "archivio_antimafia": 7, "dice": 10363, "rizzo_synthetic": 155}
assert counts == expected, counts
print("Prepared documents:", len(documents))
print(dict(counts))
del documents


## Run the expanded corpus

The corpus contains 10,651 documents. spaCy processes batches of eight with the same enabled components as run 04. Progress is printed every 100 documents and forwarded to the notebook. If interrupted, the cell terminates its child process. Use a new output name for each experiment; an existing file is protected.


In [ ]:
output_path = Path("outputs/05_batched_form_boundaries_spacy_lg.json")
command = [sys.executable, "-u", "scripts/run_pipeline.py", "--output", str(output_path)]

# Forward script output to the notebook while processing is in progress.
with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True) as process:
    try:
        for line in process.stdout:
            print(line, end="", flush=True)
        return_code = process.wait()
    except KeyboardInterrupt:
        process.terminate()
        process.wait()
        raise
    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, command)


In [ ]:
results = json.loads(output_path.read_text(encoding="utf-8"))
run = results["run"]
print("Mode:", run["mode"])
print("Model:", run["model_name"], run["model_version"])
print("Commit:", run["git"])
print("Documents:", run["document_count"])
print("Candidates:", run["candidate_count"])
print("Extraction seconds:", round(run["extraction_seconds"], 2))
print("Batch size:", run["spacy_batch_size"])
print("Pipeline:", run["spacy_pipeline"])

for source, summary in run["by_source"].items():
    print("\nSource:", source)
    print(summary)

for source in run["by_source"]:
    document = next(d for d in results["documents"] if d.get("source") == source)
    print("\nExample:", source, document["document"])
    for entity in document["entities"][:8]:
        print(entity["label"], repr(entity["text"]), entity["review_reasons"])

# Inspect the form fields affected by the boundary correction.
for document_id in ("redit:file-0025.xlsx.tsv", "redit:file-0116.xlsx.tsv"):
    document = next(d for d in results["documents"] if d["id"] == document_id)
    print("\nForm check:", document_id)
    for entity in document["entities"]:
        if entity["label"] == "PERSON":
            print(repr(entity["text"]), entity["review_reasons"])


## Save the result

Download `outputs/05_batched_form_boundaries_spacy_lg.json` manually from the Colab file panel. Once saved, disconnect the runtime while inspecting results or editing code locally. Candidate counts and flags are diagnostic outputs, not precision, recall or F1. Synthetic results must remain separate from real-document evaluation.
